# Bayesian networks with pyAgrum

**Scenario.** A university gym has a **weight room** and, on the other side of a shared wall, a
**group fitness room** used for spinning, zumba and pilates classes. The weight room has three binary
sensors, sampled every five minutes: a microphone that reports whether the sound level exceeds a
threshold, a CO&#8322; sensor that reports whether the concentration exceeds a threshold, and an
accelerometer on the floor slab that reports the rhythmic vibration of a class moving in step
(isolated impacts, such as a dropped weight, do not trigger it).

Two situations are of interest, and neither is measured directly: **`GroupClass`**, a class is running
in the group fitness room, and **`WeightRoomCrowded`**, at least four people are training in the weight
room. The system has to infer both from the three sensors. The microphone is in the weight room, but it
also picks up the sound of a class in the group fitness room.

In this notebook you define the structure of the network, assign its conditional probabilities, compute
the posterior probability of the two situations as the readings are entered, and then estimate the same
probabilities from two weeks of logged data.

In [ ]:
#@title Setup: installs pyAgrum, downloads the log and defines the helper functions. Run it first.
!pip install -q pyagrum

import os, itertools, urllib.request
import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

NODES = ["GroupClass", "WeightRoomCrowded", "WeightRoomSound", "WeightRoomCO2", "WeightRoomVibration"]
LOG_URL = ("https://raw.githubusercontent.com/lucregrassi/ambient-intelligence-labs"
           "/main/lab4-bayesian-networks/gym_log.csv")

if not os.path.exists("gym_log.csv"):
    urllib.request.urlretrieve(LOG_URL, "gym_log.csv")
log = pd.read_csv("gym_log.csv")

def new_network():
    """Five binary nodes, no arcs yet."""
    bn = gum.BayesNet("The weight room")
    for n in NODES:
        bn.add(gum.LabelizedVariable(n, n, ["no", "yes"]))
    return bn

def parents_of(bn, node):
    return list(bn.cpt(node).names[1:])

def _condition(key):
    """'GroupClass=yes' or ('GroupClass=yes', 'WeightRoomCrowded=no') -> {'GroupClass': 'yes', ...}"""
    keys = (key,) if isinstance(key, str) else key
    return dict(k.split("=") for k in keys)

def fill(bn, P):
    """Put your numbers into the network. Every number is P(node = yes)."""
    bn.cpt("GroupClass").fillWith([1 - P["GroupClass"], P["GroupClass"]])
    bn.cpt("WeightRoomCrowded").fillWith([1 - P["WeightRoomCrowded"], P["WeightRoomCrowded"]])
    for node in ("WeightRoomSound", "WeightRoomVibration"):
        for key, v in P[node].items():
            bn.cpt(node)[_condition(key)] = [1 - v, v]
    for key, v in P["WeightRoomCO2"].items():
        cond = _condition(key)
        if "GroupClass" in parents_of(bn, "WeightRoomCO2"):    # optional arc: the same guess on both sides
            for c in ("no", "yes"):
                bn.cpt("WeightRoomCO2")[dict(cond, **{"GroupClass": c})] = [1 - v, v]
        else:
            bn.cpt("WeightRoomCO2")[cond] = [1 - v, v]

def show(bn, size="5"):
    gnb.showBN(bn, size=size)

def show_cpts(bn):
    gnb.sideBySide(*[bn.cpt(n) for n in NODES], captions=NODES)

def ask(bn, evidence=None, size="8"):
    gnb.showInference(bn, evs=evidence or {}, size=size)

def belief(bn, evidence=None):
    ie = gum.LazyPropagation(bn)
    ie.setEvidence(evidence or {})
    ie.makeInference()
    return {n: round(100 * float(ie.posterior(n).toarray()[1]), 1) for n in ("GroupClass", "WeightRoomCrowded")}

def as_text(evidence):
    return ", ".join(f"{k}={v}" for k, v in evidence.items()) or "nothing observed"

def one_clue_at_a_time(bn, clues):
    """clues: a list of (node, value). Adds them one by one and reports the two situations."""
    rows, ev = [], {}
    rows.append({"what the system knows": as_text(ev), **belief(bn, ev)})
    for node, value in clues:
        ev = dict(ev, **{node: value})
        rows.append({"what the system knows": as_text(ev), **belief(bn, ev)})
    return pd.DataFrame(rows).set_index("what the system knows")

def learn(bn, n_cases):
    """Count the cases in the log and turn the counts into CPTs. Same structure, new numbers."""
    log.head(n_cases).to_csv("_subset.csv", index=False)
    learner = gum.BNLearner("_subset.csv", bn)
    learner.useSmoothingPrior(1)
    return learner.learnParameters(bn.dag())

def compare_cpt(bn_you, bn_log, node):
    parents = parents_of(bn_you, node)
    rows = []
    for combo in itertools.product(["no", "yes"], repeat=len(parents)):
        ev = dict(zip(parents, combo))
        row = {p: v for p, v in ev.items()}
        row[f"P({node}=yes) — you"] = round(float(bn_you.cpt(node)[ev][1]), 3)
        row[f"P({node}=yes) — the log"] = round(float(bn_log.cpt(node)[ev][1]), 3)
        rows.append(row)
    return pd.DataFrame(rows)

def compare_answers(bn_you, bn_log, evidence):
    return pd.DataFrame([{"": "your numbers", **belief(bn_you, evidence)},
                         {"": "numbers from the log", **belief(bn_log, evidence)}]).set_index("")

## 1. The structure

Before running any code, draw the network on paper with your group: the five binary nodes `GroupClass`,
`WeightRoomCrowded`, `WeightRoomSound`, `WeightRoomCO2`, `WeightRoomVibration`, and the arrows between
them. Each arrow goes from a cause to the variable it affects.

<details>
<summary><b>Solution (open it after drawing your own)</b></summary>

`WeightRoomSound` has two parents, `GroupClass` and `WeightRoomCrowded`, because the microphone responds
to both rooms. `WeightRoomCO2` has one parent, `WeightRoomCrowded`, because the group fitness room has its
own ventilation. `WeightRoomVibration` has one parent, `GroupClass`. There is no arrow between `GroupClass`
and `WeightRoomCrowded`: the model assumes that a class next door does not affect the number of people in
the weight room. This is a modelling assumption about this building, and it can be discussed.
</details>

In [ ]:
bn = new_network()                                      # five binary nodes, no arcs yet
bn.addArc("GroupClass",        "WeightRoomSound")       # sound of the class through the wall
bn.addArc("WeightRoomCrowded", "WeightRoomSound")       # sound of the people training in the room
bn.addArc("WeightRoomCrowded", "WeightRoomCO2")         # CO2 produced by the occupants of the room
bn.addArc("GroupClass",        "WeightRoomVibration")   # floor vibration from a class in step
# bn.addArc("GroupClass",      "WeightRoomCO2")         # optional arc, tested in section 5
show(bn)

`WeightRoomSound` is a **collider**: a node with two parents. Section 4 shows how evidence propagates
through it.

## 2. The conditional probability tables

The structure determines which parameters are needed: one prior for `GroupClass`, one for
`WeightRoomCrowded`, four entries for `WeightRoomSound` (one per combination of its two parents), two for
`WeightRoomCO2` and two for `WeightRoomVibration`. That is **ten parameters**, against 31 for the full joint
distribution of five binary variables.

The values below are not measured: they are estimates that someone familiar with the building could give.
The entry marked with an arrow is P(`WeightRoomSound` = yes | class, no crowd), the probability that the
microphone exceeds its threshold when only the class is running. Keep these values until section 5, where
they are compared with values estimated from data.

In [ ]:
P = {                                              # every number is P(the node = yes)
    "GroupClass":        0.30,                     # a class runs in about 30% of the opening hours
    "WeightRoomCrowded": 0.40,                     # four or more people in the weight room

    "WeightRoomSound": {
        ("GroupClass=yes", "WeightRoomCrowded=no"):  0.80,   # <- sound of the class through the wall
        ("GroupClass=yes", "WeightRoomCrowded=yes"): 0.97,
        ("GroupClass=no",  "WeightRoomCrowded=yes"): 0.70,
        ("GroupClass=no",  "WeightRoomCrowded=no"):  0.05,   # false positives of the microphone
    },

    "WeightRoomCO2":       {"WeightRoomCrowded=yes": 0.75, "WeightRoomCrowded=no": 0.05},
    "WeightRoomVibration": {"GroupClass=yes":        0.65, "GroupClass=no":        0.03},
}
fill(bn, P)
show_cpts(bn)

## 3. A single reading

The microphone exceeds its threshold. No other reading is available.

In [ ]:
ask(bn, {"WeightRoomSound": "yes"})

Observed nodes are shown in orange; the bars on the other nodes are their posterior probabilities. Both
situations become more likely, since either of them can explain the sound.

## 4. Adding the readings one at a time

The other two sensors are now entered one at a time. The table gives the posterior probability (in %) of
the two situations after each reading.

In [ ]:
one_clue_at_a_time(bn, [("WeightRoomSound", "yes"), ("WeightRoomCO2", "no"), ("WeightRoomVibration", "yes")])

Third row: `WeightRoomCO2` has no direct connection to `GroupClass`; the only path between them goes
through `WeightRoomCrowded` and `WeightRoomSound`. Nevertheless `GroupClass` rises from 55 to 71. The
sound is observed, so it must be explained by at least one of its two causes; a normal CO&#8322; reading
makes `WeightRoomCrowded` less likely, and the probability shifts to `GroupClass`.

Last row: `WeightRoomCrowded` ends at **18.5%, below its prior of 40%**, although none of the three sensors
measures the number of people. Once the vibration makes `GroupClass` almost certain, the class alone
accounts for the sound, which no longer supports `WeightRoomCrowded`. This effect is called **explaining
away**. It occurs only at a collider, when the collider or one of its descendants is observed.

<details>
<summary><b>Question: what happens with Vibration = no instead?</b></summary>

`WeightRoomCrowded` increases instead of decreasing. With `Vibration = no`, `GroupClass` becomes less
likely, while the sound still has to be explained, so the probability shifts back to `WeightRoomCrowded`.
To check it, replace `"yes"` with `"no"` in the cell above. The direction of the change follows from the
structure alone; the parameters determine its size.
</details>

## 5. Estimating the tables from data

`gym_log.csv` is a synthetic dataset generated for this lab, in the format of a real log: two weeks of
operation, one row per five-minute slot from 7:00 to 22:00, 2 520 rows in total. The three sensor columns
are produced by the sensors. The two situation columns (the labels) come from other sources: `GroupClass`
from the booking system (scheduled class and instructor check-in), and `WeightRoomCrowded` from a
camera-based people counter installed over the weight room door for the two weeks of data collection only
(the turnstiles count the whole building, not this room).

The labels are the costly part of the process, because they do not come from the sensors the system will
use. They are collected once, used to estimate the tables, and the system then runs on the sensors alone.

In [ ]:
log.head()

In [ ]:
N_CASES = len(log)             # every slot in the log; try 50, then 200, then len(log) again
bn_log = learn(bn, N_CASES)
compare_cpt(bn, bn_log, "WeightRoomSound")

The largest difference is in the entry marked with the arrow: 0.80 in our estimate, 0.55 in the data. The
other three entries are close.

Now set `N_CASES = 50` and run the cell again. The estimates change by tens of percentage points (the
marked entry drops to 0.43), because each one is now computed from a few rows; the first fifty slots cover
a single morning. Then set `N_CASES = len(log)` again and rerun the cell before going on. (Before counting,
the estimation adds one pseudo-count to every configuration, so that a combination never observed does not
receive a probability of 0 or 1.)

In [ ]:
compare_cpt(bn, bn_log, "WeightRoomCO2")

Two rows, since `WeightRoomCO2` has a single parent. When the weight room is crowded, the CO&#8322; sensor
exceeds its threshold in 47% of the slots, against the 0.75 assumed in section 2.

Optional: go back to section 1, **uncomment the `GroupClass -> WeightRoomCO2` arrow**, run sections 1, 2
and 5 again, and look at this table again. It now has four rows, and the two rows with
`WeightRoomCrowded = yes` read about 0.46 and 0.49. With about 1 100 rows behind them, the difference is
within sampling noise: the data do not support a dependence of the weight room CO&#8322; on the class next
door. Then comment the arrow out again and rerun sections 1, 2 and 5.

## 6. The same evidence, two sets of parameters

Evidence: sound above threshold, normal CO&#8322;, no vibration. In the log, this is the most frequent
combination when the sound is above threshold.

In [ ]:
EVIDENCE = {"WeightRoomSound": "yes", "WeightRoomCO2": "no", "WeightRoomVibration": "no"}
compare_answers(bn, bn_log, EVIDENCE)

In [ ]:
ask(bn_log, EVIDENCE)

With the values of section 2 the two situations are equally likely: 46% and 46%. With the values
estimated from the log the network favours one explanation: no class next door (29%), weight room
crowded (69%).

The two networks have the same structure and receive the same evidence. **The structure determines which
variables depend on which; the parameters determine the answer, especially when the evidence is
ambiguous.**

## 7. Where the arrows come from

Each of the four arrows corresponds to a fact about the building that could be stated in an ontology.

| fact about the building | arrow |
|---|---|
| the microphone is in the weight room, where people train | `WeightRoomCrowded` &rarr; `WeightRoomSound` |
| the group fitness room, where the class takes place, shares a wall with the weight room | `GroupClass` &rarr; `WeightRoomSound` |
| the two rooms share the same floor slab, and a class in step makes it vibrate | `GroupClass` &rarr; `WeightRoomVibration` |
| the CO&#8322; sensor is in the weight room; the group fitness room has separate ventilation | `WeightRoomCrowded` &rarr; `WeightRoomCO2`, and **no** arrow from `GroupClass` |

A general rule follows. An activity that takes place in a room becomes a hidden node; a sensor installed
in a room becomes an observed node; an arrow connects them when the activity can affect the sensor, either
in the same room or through a physical connection between rooms. Two of the four arrows cross a room
boundary (a shared wall and a shared floor slab), so the ontology must also describe which rooms are
adjacent, not only what each room contains.

**An ontology can determine which arrows are allowed; it cannot provide the values of the conditional
probabilities.** The two models are complementary: the ontology defines when a room is crowded (at least
four people), the network estimates how likely it is that the room is crowded, given the sensor readings.

## Summary

1. Each arrow adds parameters: with two parents, `WeightRoomSound` needs four entries instead of two.
2. When a collider is observed, its parents become dependent: a CO&#8322; reading changed the probability
   of a class in another room.
3. Parameters estimated from data replace subjective estimates, provided there are enough cases: with
   fifty rows the same estimates change by tens of percentage points.